# Music Dataset Exploration

## Objective

The objective of this notebook is to understand the structure, quality, and characteristics of the music dataset before building the recommendation system.

The analysis focuses on:

- Understanding the dataset structure and feature types
- Identifying missing and duplicate records
- Examining track and artist uniqueness
- Understanding the distribution of music-related features
- Identifying potential data-quality issues
- Identifying features that may be useful for recommendation
- Defining initial data-cleaning and feature-engineering considerations

In [ ]:
import pandas as pd
import numpy as np

## 1. Dataset Overview

The dataset contains Spotify music tracks together with track metadata, musical/audio characteristics, and other track-related information.

The variables can broadly be grouped into:

- **Track metadata:** track name, artist, track ID, release date, and year
- **Continuous audio features:** acousticness, danceability, energy, instrumentalness, liveness, loudness, speechiness, tempo, and valence
- **Musical features:** key and mode
- **Other track attributes:** popularity, explicit content, and duration

In [ ]:
DATA_PATH = "../data/raw/data.csv"

df = pd.read_csv(DATA_PATH)

df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe(include = 'all').T

### Initial Observation

The dataset contains approximately 170K rows and 19 columns before data preprocessing.

The dataset contains a mixture of continuous numerical features, binary/discrete musical features, and textual metadata. Therefore, these different feature types may require different preprocessing strategies.

### Data Type Observations

Pandas reports mixed data types in the `artists`, `id`, `name`, and `release_date` columns.

This indicates that these columns require additional inspection before defining the final preprocessing pipeline.

In particular:

- `id` should be treated as an identifier rather than a numerical feature.
- `name` and `artists` represent metadata.
- `release_date` requires further inspection because it may contain different date representations.
- Audio features should be treated separately from metadata columns.

## 2. Data Quality

### Missing Values

The dataset contains rows where all columns are missing.

These rows do not provide any useful information for recommendation or analysis and should therefore be removed during the preprocessing stage.

The remaining records contain the actual music-related information that will be used for further analysis.

In [ ]:
df.isna()

In [ ]:
df.isna().sum()

In [ ]:
df.tail(13163).isna().sum()

### Finding

There are 13,163 rows where all 19 columns contain missing values.

### Decision

These completely empty rows will be removed during data preprocessing.

The cleaning operation will be implemented separately from this exploratory notebook so that this notebook preserves the original dataset exploration.

## 3. Duplicate and Uniqueness Analysis

In [ ]:
df.duplicated().sum()

In [ ]:
df.nunique()

In [ ]:
df['mode'].value_counts()

In [ ]:
df[['name', 'artists']].head(10)

In [ ]:
df['name'].nunique()

In [ ]:
df['name'].value_counts()

In [ ]:
len(df['name'].value_counts()[df['name'].value_counts() > 1])

In [ ]:
df[df['name'] == 'White Christmas'].head()

In [ ]:
df['artists'].nunique()

In [ ]:
df['artists'].value_counts()

In [ ]:
len(df['artists'].value_counts()[df['artists'].value_counts() > 1])

In [ ]:
df['artists'].value_counts()[df['artists'].value_counts() > 1]

In [ ]:
df['name'].duplicated()

In [ ]:
df['name'].duplicated().sum()

### Track ID

The `id` column contains 157,490 non-null values and 157,490 unique values.

This indicates that the available non-null track IDs are unique and can serve as the primary identifier for tracks in the usable dataset.

### Track Name

The `name` column contains 124,353 unique values among 157,490 non-null values.

Therefore, track name alone cannot be used as a unique identifier.

For example, the track name `White Christmas` occurs multiple times in the dataset.

### Artist

The `artists` column contains approximately 32,000 unique values among the non-null records.

This indicates that the dataset contains a large number of distinct artist values.

### Interpretation

`duplicated()` and `nunique()` provide different information.

- `duplicated()` identifies rows that duplicate another complete row.
- `nunique()` counts distinct values within a particular column.

Therefore, repeated values in `name` or `artists` do not necessarily indicate duplicate tracks.

Track identity should primarily be based on the track `id` rather than the track name.

## 4. Feature Categories

The variables in the dataset represent different types of information about a track. They should therefore be considered according to their semantic meaning rather than simply their Pandas data type.

### 4.1 Continuous Audio Features

The dataset contains the following continuous audio features:

- `acousticness`
- `danceability`
- `energy`
- `instrumentalness`
- `liveness`
- `loudness`
- `speechiness`
- `tempo`
- `valence`

These features describe continuous characteristics of a track and can form the core of an audio-based track representation for similarity-based recommendation.

### 4.2 Musical Features

The dataset also contains `key` and `mode`.

`key` represents the musical key of a track and takes discrete values.

`mode` represents the modality of the track and contains two possible values, 0 and 1.

Although `mode` is stored as a numerical variable, it should be treated as a binary musical feature rather than as a continuous numerical measurement.

The appropriate representation and whether these features should be included in the final similarity representation will be investigated during feature engineering.

In [ ]:
# Features requireed further numerical analysis.
features = [
    "acousticness",
    "danceability",
    "energy",
    "instrumentalness",
    "liveness",
    "loudness",
    "speechiness",
    "valence",
    "tempo",
    "duration_ms",
    "key",
    "mode"
]

In [ ]:
df[features].info()

In [ ]:
df[features].describe().T

In [ ]:
df[features].isna()

In [ ]:
df[features].isna().sum()

### Correlation Analysis

Pearson correlation is useful for examining linear relationships between the continuous numerical features.

The correlation matrix below also includes `duration_ms`, `key`, and `mode` for exploratory purposes. However, `key` and `mode` are discrete musical features and their correlation coefficients should not be interpreted in the same way as correlations between continuous audio features.

The correlation analysis is therefore treated as exploratory rather than as a basis for deciding which features should be included in the final recommendation representation.

In [ ]:
df[features].corr()

### Final Summary

- The dataset consists of approximately 170K records and 19 attributes.
- The dataset contains 13,163 completely null rows, which are located at the tail of the dataset.
- The columns `artists`, `id`, `name`, and `release_date` are represented as string/object-like data, while the remaining columns are represented numerically in the loaded DataFrame.
- The `artists` values represent lists of artist names stored as strings in the dataset.
- The features `acousticness`, `danceability`, `energy`, `instrumentalness`, `liveness`, `speechiness`, `valence`, `loudness`, and `tempo` represent continuous audio characteristics.
- `duration_ms` represents track duration and is a continuous numerical measurement, but it has a different semantic role from the core audio characteristics.
- `key` is a discrete musical feature.
- `mode` is a binary musical feature containing values 0 and 1.
- Track `id` values are unique among the non-null records and should be used as track identifiers rather than as modeling features.
- Track names are not unique and therefore should not be used as the primary track identifier.
- Further preprocessing and feature engineering are required before constructing the recommendation model.